# Real-Time Dashboard (Section 2.4)
This notebook builds and launches a live Streamlit dashboard that reads Parquet data from Azure Blob Storage and visualizes real-time food delivery analytics.

**Run all cells from top to bottom to launch the dashboard.**

## Cell 1 — Install dependencies
Installs all required Python libraries: Streamlit for the dashboard, azure-storage-blob to read from Azure, and plotly for interactive charts.

In [2]:
!pip install streamlit pyngrok azure-storage-blob plotly -q

## Cell 2 — Explore Blob Storage structure
Connects to the Azure Blob Storage account and lists all files in the `group10output` container. This confirms the Parquet files written by Spark Structured Streaming are available, partitioned by `event_date` and `zone_id`.

In [3]:
from azure.storage.blob import BlobServiceClient

from google.colab import userdata
STORAGE_KEY = userdata.get("STORAGE_KEY")  # Colab Secret — never hardcode keys
CONNECTION_STRING = f"DefaultEndpointsProtocol=https;AccountName=iesstsabbadbaa;AccountKey={STORAGE_KEY};EndpointSuffix=core.windows.net"
CONTAINER_NAME = "group10output"

client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container = client.get_container_client(CONTAINER_NAME)

for blob in container.list_blobs():
    print(blob.name)

bronze
bronze/couriers
bronze/couriers/_spark_metadata
bronze/couriers/_spark_metadata/0
bronze/couriers/event_date=2026-04-17
bronze/couriers/event_date=2026-04-17/zone_id=zone_centre
bronze/couriers/event_date=2026-04-17/zone_id=zone_centre/part-00002-2a661c64-1aeb-463c-872d-6348f9c17e06.c000.snappy.parquet
bronze/couriers/event_date=2026-04-17/zone_id=zone_east
bronze/couriers/event_date=2026-04-17/zone_id=zone_east/part-00002-7f1daf56-0c92-42d2-8a6e-4a1c3eb886c6.c000.snappy.parquet
bronze/couriers/event_date=2026-04-17/zone_id=zone_north
bronze/couriers/event_date=2026-04-17/zone_id=zone_north/part-00000-95ffb5a3-4864-445f-baff-05c8679d9b76.c000.snappy.parquet
bronze/couriers/event_date=2026-04-17/zone_id=zone_south
bronze/couriers/event_date=2026-04-17/zone_id=zone_south/part-00001-d301324b-9e7a-41f6-9a9c-00f0b553bd18.c000.snappy.parquet
bronze/couriers/event_date=2026-04-17/zone_id=zone_suburbs
bronze/couriers/event_date=2026-04-17/zone_id=zone_suburbs/part-00000-93bdba79-7936-45

## Cell 3 — Inspect data schema
Loads one sample Parquet file from each feed (orders and couriers) to verify the column names and data types before building the dashboard. This ensures the dashboard uses the correct field names.

In [4]:
import pandas as pd
from azure.storage.blob import BlobServiceClient
from io import BytesIO

from google.colab import userdata
STORAGE_KEY = userdata.get("STORAGE_KEY")  # Colab Secret — never hardcode keys
CONNECTION_STRING = f"DefaultEndpointsProtocol=https;AccountName=iesstsabbadbaa;AccountKey={STORAGE_KEY};EndpointSuffix=core.windows.net"
CONTAINER_NAME = "group10output"

client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container = client.get_container_client(CONTAINER_NAME)

# Load one orders file
orders_blob = "bronze/orders/event_date=2026-04-18/zone_id=zone_centre/part-00002-8c886081-0346-46b9-9aa7-af7bb4da4426.c000.snappy.parquet"
data = container.get_blob_client(orders_blob).download_blob().readall()
orders_df = pd.read_parquet(BytesIO(data))

print("=== ORDERS COLUMNS ===")
print(orders_df.columns.tolist())
print("\n=== SAMPLE ROW ===")
print(orders_df.head(2))

# Load one couriers file
couriers_blob = "bronze/couriers/event_date=2026-04-18/zone_id=zone_centre/part-00002-692f8dc7-2ba5-42c6-84f9-f7907f5e598a.c000.snappy.parquet"
data2 = container.get_blob_client(couriers_blob).download_blob().readall()
couriers_df = pd.read_parquet(BytesIO(data2))

print("\n=== COURIERS COLUMNS ===")
print(couriers_df.columns.tolist())
print("\n=== SAMPLE ROW ===")
print(couriers_df.head(2))

=== ORDERS COLUMNS ===
['event_id', 'order_id', 'customer_id', 'restaurant_id', 'courier_id', 'event_type', 'event_time', 'ingestion_time', 'order_value', 'promo_code', 'promo_discount_pct', 'estimated_prep_min', 'is_late_event', 'is_duplicate', 'schema_version', 'event_time_ts']

=== SAMPLE ROW ===
                               event_id                              order_id  \
0  16abd795-17ad-47ae-9431-74dd34c69ac7  afeb8650-55f8-4f89-90e3-71aaabe115f7   
1  6a4122eb-f42e-4f35-9204-746506bb1c42  afeb8650-55f8-4f89-90e3-71aaabe115f7   

  customer_id restaurant_id    courier_id        event_type  \
0  cust_00031     rest_0004          None  READY_FOR_PICKUP   
1  cust_00031     rest_0004  courier_0048         PICKED_UP   

                 event_time            ingestion_time  order_value promo_code  \
0  2026-04-18T01:14:11.435Z  2026-04-18T01:14:11.763Z    35.009998       None   
1  2026-04-18T01:22:11.435Z  2026-04-18T01:22:11.994Z    35.009998       None   

   promo_discount_pct

## Cell 4 — Write the dashboard file
Writes `dashboard.py` to disk. This is the full Streamlit application that:
- Loads all Parquet files from both the `orders` and `couriers` bronze layer
- Provides sidebar filters by **zone** and **restaurant**
- Displays live KPIs, time-series charts, event type breakdowns, courier status, speed/battery distributions, and a health & alerts section
- Caches data for 30 seconds to simulate a live refresh

In [5]:
dashboard_code = '''
import streamlit as st
import pandas as pd
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import plotly.express as px

import os
STORAGE_KEY = os.environ["STORAGE_KEY"]  # passed in from the Colab Secret at launch
CONNECTION_STRING = f"DefaultEndpointsProtocol=https;AccountName=iesstsabbadbaa;AccountKey={STORAGE_KEY};EndpointSuffix=core.windows.net"
CONTAINER_NAME = "group10output"

@st.cache_data(ttl=30)
def load_all_parquet(prefix):
    client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container = client.get_container_client(CONTAINER_NAME)
    frames = []
    for blob in container.list_blobs(name_starts_with=prefix):
        if blob.name.endswith(".parquet"):
            data = container.get_blob_client(blob.name).download_blob().readall()
            frames.append(pd.read_parquet(BytesIO(data)))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

st.set_page_config(page_title="Live Delivery Analytics", layout="wide")
st.title("🚗 Real-Time Food Delivery Dashboard")

orders_df = load_all_parquet("bronze/orders/")
couriers_df = load_all_parquet("bronze/couriers/")

orders_df["event_time_ts"] = pd.to_datetime(orders_df["event_time_ts"])
couriers_df["event_time_ts"] = pd.to_datetime(couriers_df["event_time_ts"])

latest_couriers = couriers_df.sort_values("event_time_ts").groupby("courier_id").last().reset_index()

# -- SIDEBAR FILTERS --
st.sidebar.header("🔍 Filters")

selected_zone = st.sidebar.selectbox("Zone", ["All"] + ["zone_centre","zone_east","zone_north","zone_south","zone_suburbs","zone_west"])

restaurants = ["All"] + sorted(orders_df["restaurant_id"].dropna().unique().tolist())
selected_restaurant = st.sidebar.selectbox("Restaurant", restaurants)

# Apply filters
filtered_orders = orders_df.copy()
filtered_couriers = couriers_df.copy()
filtered_latest = latest_couriers.copy()

if selected_zone != "All":
    filtered_orders = filtered_orders[filtered_orders["zone_id"] == selected_zone] if "zone_id" in filtered_orders.columns else filtered_orders
    filtered_couriers = filtered_couriers[filtered_couriers["zone_id"] == selected_zone] if "zone_id" in filtered_couriers.columns else filtered_couriers
    filtered_latest = filtered_latest[filtered_latest["zone_id"] == selected_zone] if "zone_id" in filtered_latest.columns else filtered_latest

if selected_restaurant != "All":
    filtered_orders = filtered_orders[filtered_orders["restaurant_id"] == selected_restaurant]

# -- KPIs --
st.subheader("📊 Live KPIs")
col1, col2, col3, col4, col5 = st.columns(5)
total_orders = filtered_orders["order_id"].nunique()
total_events = len(filtered_orders)
late_events = filtered_orders["is_late_event"].sum()
duplicates = filtered_orders["is_duplicate"].sum()
active_couriers = filtered_latest["courier_id"].nunique()
avg_order_value = filtered_orders["order_value"].mean()
col1.metric("Unique Orders", total_orders)
col2.metric("Total Events", total_events)
col3.metric("Active Couriers", active_couriers)
col4.metric("⚠️ Late Events", int(late_events))
col5.metric("💰 Avg Order Value", f"€{avg_order_value:.2f}" if not pd.isna(avg_order_value) else "—")

st.divider()

# -- TIME SERIES --
st.subheader("📈 Order Events Over Time")
ts = filtered_orders.set_index("event_time_ts").resample("10min").size().reset_index(name="count")
fig1 = px.line(ts, x="event_time_ts", y="count", title="Order Events per 10-minute window")
st.plotly_chart(fig1, use_container_width=True)

st.divider()

# -- EVENT TYPE + COURIER STATUS --
col_a, col_b = st.columns(2)
with col_a:
    st.subheader("📦 Orders by Event Type")
    event_counts = filtered_orders.groupby("event_type").size().reset_index(name="count")
    fig2 = px.bar(event_counts, x="event_type", y="count", color="event_type")
    st.plotly_chart(fig2, use_container_width=True)
with col_b:
    st.subheader("🗺️ Courier Status Distribution")
    status_counts = filtered_latest.groupby("status").size().reset_index(name="count")
    fig3 = px.pie(status_counts, names="status", values="count")
    st.plotly_chart(fig3, use_container_width=True)

st.divider()

# -- SPEED + BATTERY (latest ping per courier) --
col_c, col_d = st.columns(2)
with col_c:
    st.subheader("🏎️ Courier Speed Distribution")
    fig4 = px.histogram(filtered_latest, x="speed_kmh", nbins=20, title="Speed (km/h) — latest ping per courier")
    st.plotly_chart(fig4, use_container_width=True)
with col_d:
    st.subheader("🔋 Courier Battery Levels")
    fig5 = px.histogram(filtered_latest, x="battery_pct", nbins=20, title="Battery % — latest ping per courier")
    st.plotly_chart(fig5, use_container_width=True)

st.divider()

# -- HEALTH & ALERTS --
st.subheader("🚨 Health & Alerts")
col_e, col_f, col_g = st.columns(3)
late_pct = (late_events / total_events * 100) if total_events > 0 else 0
dup_pct = (duplicates / total_events * 100) if total_events > 0 else 0
low_battery_couriers = int((filtered_latest["battery_pct"] < 20).sum())
col_e.metric("Late Event Rate", f"{late_pct:.1f}%")
col_f.metric("Duplicate Event Rate", f"{dup_pct:.1f}%")
col_g.metric("🔋 Low Battery Couriers (<20%)", low_battery_couriers)

if late_pct > 10:
    st.error(f"⚠️ High late event rate: {late_pct:.1f}%!")
elif late_pct > 5:
    st.warning(f"Late event rate elevated: {late_pct:.1f}%")
else:
    st.success("✅ Late event rate is within normal range.")

if low_battery_couriers > 10:
    st.warning(f"⚠️ {low_battery_couriers} couriers have battery below 20% — consider recharging alerts.")

st.caption(f"Data refreshes every 30 seconds. Last loaded: {pd.Timestamp.now().strftime('%H:%M:%S')}")
'''

with open("dashboard.py", "w") as f:
    f.write(dashboard_code)

print("dashboard.py written successfully!")

dashboard.py written successfully!


## Cell 5 — Launch the dashboard
Downloads `cloudflared` to create a public tunnel, starts Streamlit on port 8501, and exposes it via a public `trycloudflare.com` URL. Open that URL in your browser to view the live dashboard.

> **Note:** The URL changes every time this cell is run. Keep this Colab session open to keep the dashboard alive.

In [6]:
import os
import subprocess
import time
from google.colab import userdata

# Hand the storage key to the Streamlit process via an environment variable
os.environ["STORAGE_KEY"] = userdata.get("STORAGE_KEY")

# Download cloudflared tunnel binary
subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)

# Start Streamlit in the background
subprocess.Popen(["streamlit", "run", "dashboard.py", "--server.port=8501", "--server.headless=true"])
time.sleep(5)

# Start cloudflared tunnel and wait for public URL
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], stderr=subprocess.PIPE)
time.sleep(10)

for _ in range(50):
    line = tunnel.stderr.readline().decode("utf-8")
    print(line, end="")
    if "trycloudflare.com" in line and "https://" in line:
        break

2026-04-19T14:37:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-19T14:37:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-19T14:37:32Z INF +--------------------------------------------------------------------------------------------+
2026-04-19T14:37:32Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-19T14:37:32Z INF |  https://millennium-advertisements-gone-pounds.tryclou